# Phase B0 — pinned manifest-only compatibility gate

This notebook generates only four JSON continuity manifests. It loads no model resources, trains no model, creates no checkpoint, does not mount Drive, and grants no recovery authorization.


## 1. Upload the exact compatibility bundle


In [ ]:
from google.colab import files
uploaded = files.upload()
archives = [name for name in uploaded if name.endswith('.tar.gz')]
if len(archives) != 1:
    raise RuntimeError('Upload exactly one manifest-compatibility .tar.gz bundle.')
ARCHIVE = archives[0]
print('Uploaded:', ARCHIVE)


## 2. Restore the exact source commit


In [ ]:
import json, pathlib, subprocess, sys, tarfile
extract_root = pathlib.Path('/content/phase_b0_manifest_compatibility_bundle')
repo_root = pathlib.Path('/content/latent-stroke-dynamics')
if extract_root.exists() or repo_root.exists():
    raise RuntimeError('Compatibility extraction already exists; use a fresh runtime.')
extract_root.mkdir()
with tarfile.open(ARCHIVE, 'r:gz') as archive:
    archive.extractall(extract_root)
manifest = json.loads((extract_root / 'bundle_manifest.json').read_text())
if manifest['status'] != 'phase_b0_colab_manifest_compatibility_bundle_unauthorized':
    raise RuntimeError('Unexpected compatibility bundle status.')
if manifest['resource_count'] != 0 or manifest['contains_model_resources'] is not False:
    raise RuntimeError('Compatibility bundle unexpectedly contains model resources.')
if manifest['recovery_authorized'] is not False or manifest['scientific_training_allowed'] is not False:
    raise RuntimeError('Compatibility bundle crossed its non-training boundary.')
subprocess.run(['git', 'clone', '--branch', manifest['branch'], str(extract_root / 'repository.bundle'), str(repo_root)], check=True)
head = subprocess.check_output(['git', '-C', str(repo_root), 'rev-parse', 'HEAD'], text=True).strip()
if head != manifest['source_commit']:
    raise RuntimeError('Restored Git commit does not match the bundle manifest.')
print(json.dumps(manifest, indent=2))


## 3. Create an isolated CPU environment with NumPy `2.5.2` and Pillow `12.3.0`


In [ ]:
venv_root = pathlib.Path('/content/phase_b0_manifest_compatibility_venv')
if venv_root.exists():
    raise RuntimeError('Compatibility environment already exists; use a fresh runtime.')
subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', str(venv_root)], check=True)
venv_python = venv_root / 'bin' / 'python'
subprocess.run([str(venv_python), '-m', 'pip', 'install', '-q', '--upgrade', 'numpy==2.5.2', 'pillow==12.3.0', 'pytest==9.1.1'], check=True)
subprocess.run([str(venv_python), '-m', 'pip', 'install', '-q', '--no-deps', '-e', str(repo_root)], check=True)


In [ ]:
environment_check = "import json, numpy, PIL, torch, platform; print(json.dumps({'python': platform.python_version(), 'numpy': numpy.__version__, 'pillow': PIL.__version__, 'torch': str(torch.__version__)}, indent=2))"
subprocess.run([str(venv_python), '-c', environment_check], check=True)


## 4. Run the eight compatibility boundary tests


In [ ]:
completed = subprocess.run([str(venv_python), '-m', 'pytest', '-q', 'tests/test_phase_b_manifest_compatibility.py'], cwd=repo_root, text=True, capture_output=True)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
if completed.returncode != 0:
    raise RuntimeError('Compatibility tests failed; stop without generating manifests.')


Expected: **8 passed**. Any failure stops before manifest generation.


## 5. Generate and compare only the four manifests


In [ ]:
output_root = pathlib.Path('/content/phase-b0-manifest-compatibility-output')
subprocess.run([str(venv_python), 'experiments/27_phase_b_colab_manifest_compatibility.py', '--output-dir', str(output_root)], cwd=repo_root, check=True)


## 6. Inspect and download all five evidence files


In [ ]:
report_path = output_root / 'manifest_compatibility_report.json'
report = json.loads(report_path.read_text())
print(json.dumps(report, indent=2, sort_keys=True))
if report['scientific_models_trained'] is not False or report['recovery_authorized'] is not False:
    raise RuntimeError('Compatibility report crossed its non-training boundary.')
if report['google_drive_accessed'] is not False or report['historical_incomplete_directories_touched'] is not False:
    raise RuntimeError('Compatibility report crossed its storage boundary.')
download_paths = [report_path] + sorted((output_root / 'data_manifests').glob('*.json'))
if len(download_paths) != 5:
    raise RuntimeError('Expected one report and exactly four manifests.')
for path in download_paths:
    print('Downloading:', path.name)
    files.download(str(path))
print('Hash gate passed:', report['hash_gate_passed'])
print('Training remains unauthorized regardless of this result.')


Send the report, four manifests, and the test output for review. Do not start any scientific run.
